In [0]:
%run /Workspace/Users/superromelus@gmail.com/SalesLT/02_clean_silver

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 5 — VALIDATION CHECK 1 : NULL on required columns
# ─────────────────────────────────────────────────────────────────────────────
# WHY : Required fields being NULL breaks every downstream join and report.
# HOW : SQL COUNT(*) WHERE col IS NULL — reported per column, per table.
 
REQUIRED = {
    "customer":           ["CustomerID", "FirstName", "LastName", "EmailAddress"],
    "product":            ["ProductID", "Name", "ProductNumber", "ListPrice"],
    "sales_order_header": ["SalesOrderID", "OrderDate", "CustomerID",
                           "SubTotal", "TaxAmt", "Freight", "TotalDue"],
    "sales_order_detail": ["SalesOrderID", "SalesOrderDetailID",
                           "ProductID", "OrderQty", "UnitPrice", "LineTotal"],
}
 
print("── VALIDATION 1 : NULL checks on required columns ──────────────")
for table, src in active_views.items():
    cols        = spark.table(src).columns
    col_lower   = {c.lower(): c for c in cols}
    req_cols    = [col_lower.get(r.lower()) for r in REQUIRED.get(table, [])
                   if col_lower.get(r.lower())]
    issues_found = False
    for col in req_cols:
        n = spark.sql(
            f"SELECT COUNT(*) AS n FROM {src} WHERE `{col}` IS NULL"
        ).collect()[0]["n"]
        if n > 0:
            print(f"  ⚠  [{table}] {col}: {n:,} NULL(s)")
            issues_found = True
    if not issues_found:
        print(f"  ✓  [{table}] all required columns are non-null")
 
print()

In [0]:
PKS = {
    "customer":           ["CustomerID"],
    "product":            ["ProductID"],
    "sales_order_header": ["SalesOrderID"],
    "sales_order_detail": ["SalesOrderID", "SalesOrderDetailID"],
}
 
print("── VALIDATION 2 : Duplicate primary keys ───────────────────────")
for table, src in active_views.items():
    cols      = spark.table(src).columns
    col_lower = {c.lower(): c for c in cols}
    pk_cols   = [col_lower.get(p.lower()) for p in PKS.get(table, [])
                 if col_lower.get(p.lower())]
    if not pk_cols:
        continue
 
    partition = ", ".join(f"`{c}`" for c in pk_cols)
    n = spark.sql(f"""
        SELECT COUNT(*) AS n FROM (
            SELECT COUNT(*) OVER (PARTITION BY {partition}) AS cnt
            FROM   {src}
        ) WHERE cnt > 1
    """).collect()[0]["n"]
 
    if n > 0:
        print(f"  ⚠  [{table}] {n:,} duplicate row(s) on {pk_cols}")
    else:
        print(f"  ✓  [{table}] no duplicate keys")
 
print()

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — VALIDATION CHECK 3 : Negative or zero amounts
# ─────────────────────────────────────────────────────────────────────────────
# WHY : Prices, quantities and totals must be positive — negatives mean bad data.
# HOW : SQL CASE WHEN col <= 0 per amount column.
 
AMOUNTS = {
    "product":            [("ListPrice",  "non_negative"),
                           ("StandardCost", "non_negative")],
    "sales_order_header": [("SubTotal",   "non_negative"),
                           ("TaxAmt",     "non_negative"),
                           ("Freight",    "non_negative"),
                           ("TotalDue",   "non_negative")],
    "sales_order_detail": [("OrderQty",   "positive"),
                           ("UnitPrice",  "non_negative"),
                           ("LineTotal",  "non_negative")],
}
 
print("── VALIDATION 3 : Negative / zero amount checks ────────────────")
for table, src in active_views.items():
    cols      = spark.table(src).columns
    col_lower = {c.lower(): c for c in cols}
    checks    = AMOUNTS.get(table, [])
    if not checks:
        print(f"  –  [{table}] no amount columns to check")
        continue
    issues_found = False
    for col_name, rule in checks:
        actual = col_lower.get(col_name.lower())
        if not actual:
            continue
        condition = f"`{actual}` < 0" if rule == "non_negative" else f"`{actual}` <= 0"
        n = spark.sql(
            f"SELECT COUNT(*) AS n FROM {src} WHERE {condition}"
        ).collect()[0]["n"]
        if n > 0:
            print(f"  ⚠  [{table}] {actual}: {n:,} invalid value(s) ({rule})")
            issues_found = True
    if not issues_found:
        print(f"  ✓  [{table}] all amount columns valid")
 
print()

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — VALIDATION CHECK 4 : Date consistency
# ─────────────────────────────────────────────────────────────────────────────
# WHY : OrderDate must be before ShipDate and DueDate — violations mean
#       pipeline or source errors.
# HOW : SQL WHERE earlier_date > later_date AND later_date IS NOT NULL
 
DATE_CHECKS = {
    "sales_order_header": [("OrderDate", "ShipDate"),
                           ("OrderDate", "DueDate")],
    "product":            [("SellStartDate", "SellEndDate")],
}
 
print("── VALIDATION 4 : Date consistency ─────────────────────────────")
for table, src in active_views.items():
    cols      = spark.table(src).columns
    col_lower = {c.lower(): c for c in cols}
    checks    = DATE_CHECKS.get(table, [])
    if not checks:
        print(f"  –  [{table}] no date pairs to check")
        continue
    issues_found = False
    for earlier, later in checks:
        e_col = col_lower.get(earlier.lower())
        l_col = col_lower.get(later.lower())
        if not e_col or not l_col:
            continue
        n = spark.sql(f"""
            SELECT COUNT(*) AS n FROM {src}
            WHERE  `{l_col}` IS NOT NULL
            AND    TO_DATE(`{e_col}`) > TO_DATE(`{l_col}`)
        """).collect()[0]["n"]
        if n > 0:
            print(f"  ⚠  [{table}] {e_col} > {l_col}: {n:,} row(s)")
            issues_found = True
    if not issues_found:
        print(f"  ✓  [{table}] all date pairs consistent")
 
print()